# Certamen 1 — Ensambles, costos y umbral óptimo

**IEEE-CIS Fraud Detection**  
**Profesor:** Danilo Gómez Correa  
**Integrantes:** Miguel González · Jorge Moscoso · Francisca Palacios

## 0. Librerías y base de datos

Modo `entrenar`: subir `train_transaction.zip`. Modo `resultados`: subir `Resultados_Certamen1.zip` para revisar gráficos y dashboard sin reentrenar.

In [ ]:
%pip -q install "scikit-learn>=1.5" "imbalanced-learn>=0.13" xgboost lightgbm joblib ipywidgets

In [ ]:
# Librerías y parámetros usados por el grupo
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (confusion_matrix, f1_score, make_scorer,
                             precision_score, recall_score, roc_auc_score,
                             brier_score_loss)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     TunedThresholdClassifierCV, cross_val_predict,
                                     train_test_split)
from sklearn.pipeline import Pipeline as PipelineSK
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SEMILLA = 2026
N_MUESTRA = 250_000
N_BUSQUEDA = 30_000
N_ITERACIONES = 10
N_FOLDS = 3
N_JOBS = 1                # para controlar el uso de memoria en Colab
C_FP = 5_000             # supuesto academico original del grupo
C_FN = 50_000            # supuesto academico original del grupo

VARIABLES = [
    "TransactionDT", "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6", "addr1",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10",
    "C11", "C12", "C13", "C14", "D1", "D10", "D15", "V95", "V130", "V279"
]
NOMBRES = ["Decision Tree", "Random Forest", "XGBoost", "AdaBoost", "LightGBM"]
ESTRATEGIAS = ["SMOTE", "Pesos"]


# Directorios de trabajo de Colab
PROYECTO = Path('/content/Certamen1_MachineLearning')
DATOS = PROYECTO / 'data'
RESULTADOS = PROYECTO / 'resultados'
for carpeta in (DATOS, RESULTADOS, RESULTADOS / 'modelos'):
    carpeta.mkdir(parents=True, exist_ok=True)
from IPython.display import display

# 'entrenar': proyecto completo desde el dataset de Kaggle.
# 'resultados': mostrar tablas, gráficos y simulador SIN repetir entrenamiento.
MODO = 'resultados'


In [ ]:
# Solo en modo resultados: cargar el ZIP generado al finalizar el Colab anterior.
from google.colab import files
from io import BytesIO
from zipfile import ZipFile

if MODO == 'resultados':
    print('Selecciona Resultados_Certamen1.zip (no el ZIP de Kaggle)')
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError('Selecciona un único ZIP de resultados.')
    nombre_archivo, contenido_zip = next(iter(subidos.items()))
    with ZipFile(BytesIO(contenido_zip)) as zip_resultados:
        for miembro in zip_resultados.infolist():
            if miembro.is_dir():
                continue
            ruta = Path(miembro.filename)
            # Solo se permiten archivos regulares de las carpetas esperadas.
            if ('..' in ruta.parts or ruta.is_absolute() or
                    not ruta.parts or ruta.parts[0] not in ('resultados', 'graficos')):
                continue
            if ruta.suffix.lower() not in ('.csv', '.json', '.png'):
                continue
            destino = (PROYECTO / ruta if ruta.parts[0] == 'resultados'
                       else RESULTADOS / ruta)
            destino.parent.mkdir(parents=True, exist_ok=True)
            with zip_resultados.open(miembro) as entrada, destino.open('wb') as salida:
                import shutil
                shutil.copyfileobj(entrada, salida)
    indispensables = ['Tabla_CV_10_combinaciones.csv', 'Comparacion_10_Modelos.csv',
                     'Comparacion_Costos.csv', 'Sensibilidad_Costos.csv',
                     'Probabilidades_Test.csv', 'Probabilidades_CV_train.csv',
                     'Configuracion_Costos.json']
    faltantes = [archivo for archivo in indispensables if not (RESULTADOS / archivo).exists()]
    if faltantes:
        raise FileNotFoundError('Archivos faltantes en el ZIP: ' + ', '.join(faltantes))
    print('Resultados recuperados. No será necesario entrenar de nuevo.')
else:
    print('Modo entrenar: continúa con la carga de train_transaction.zip.')

In [ ]:
if MODO == 'entrenar':
    # Subir el archivo ZIP de Kaggle (también acepta CSV)
    from google.colab import files
    from io import BytesIO
    from zipfile import ZipFile, is_zipfile
    import shutil

    archivo_csv = DATOS / 'train_transaction.csv'
    if not archivo_csv.exists():
        print('Selecciona train_transaction.zip')
        subida = files.upload()
        if len(subida) != 1:
            raise ValueError('Sube solo el archivo del dataset.')
        nombre, contenido = next(iter(subida.items()))
        flujo = BytesIO(contenido)
        if is_zipfile(flujo):
            flujo.seek(0)
            with ZipFile(flujo) as comprimido:
                candidatos = [a for a in comprimido.infolist()
                              if not a.is_dir() and Path(a.filename).name == 'train_transaction.csv']
                if len(candidatos) != 1:
                    raise ValueError('No se encontró train_transaction.csv dentro del ZIP.')
                with comprimido.open(candidatos[0]) as origen, archivo_csv.open('wb') as destino:
                    shutil.copyfileobj(origen, destino)
        elif nombre.lower().endswith('.csv'):
            archivo_csv.write_bytes(contenido)
        else:
            raise ValueError('Selecciona un ZIP o CSV válido.')
    print('Dataset:', archivo_csv)
else:
    print('Se omite: resultados recuperados del ZIP.')

In [ ]:
if MODO == 'entrenar':
    # Revisión rápida del archivo
    columnas = pd.read_csv(archivo_csv, nrows=0).columns
    objetivo = pd.read_csv(archivo_csv, usecols=['isFraud'])
    print('Registros:', len(objetivo))
    print('Variables:', len(columnas))
    print('Fraudes:', int(objetivo['isFraud'].sum()))
    assert (len(objetivo), len(columnas), int(objetivo['isFraud'].sum())) == (590540, 394, 20663)
    del objetivo
else:
    print('Se omite: resultados recuperados del ZIP.')

## 1. Análisis inicial de la base

In [ ]:
def carpetas(prueba=False):
    salida = PROYECTO / ('prueba_rapida' if prueba else 'resultados')
    modelos = salida / 'modelos'
    for ruta in (DATOS, salida, modelos):
        ruta.mkdir(parents=True, exist_ok=True)
    return DATOS, salida, modelos

def parametros_ejecucion(prueba=False):
    if prueba:
        print("\n*** MODO PRUEBA: NO UTILIZAR ESTOS RESULTADOS EN LA ENTREGA ***")
        return 4_000, 800, 2, 2
    return N_MUESTRA, N_BUSQUEDA, N_ITERACIONES, N_FOLDS

def cargar_datos(prueba=False):
    """1. Problema y base. 2. Exploracion y particion 80/20.

    La lectura selectiva de columnas es una optimizacion de memoria, NO una
    limpieza aprendida. Imputaciones y codificacion ocurren en cada fold.
    """
    datos_dir, salida, _ = carpetas(prueba)
    archivo = datos_dir / "train_transaction.csv"
    if not archivo.is_file():
        raise FileNotFoundError(f"Falta {archivo}; debe ser el CSV extraido (no un ZIP).")
    columnas_originales = list(pd.read_csv(archivo, nrows=0).columns)
    esperadas = VARIABLES + ["isFraud", "TransactionID"]
    faltantes = sorted(set(esperadas) - set(columnas_originales))
    if faltantes:
        raise ValueError(f"Faltan columnas: {faltantes}. ¿Se descomprimio el CSV?")

    print("\n1. PROBLEMA DE NEGOCIO Y BASE IEEE-CIS")
    print("Columnas del archivo original:", len(columnas_originales))
    df = pd.read_csv(archivo, usecols=esperadas, low_memory=False)
    print("Registros originales:", len(df))
    print("Fraudes originales:", int(df["isFraud"].sum()))
    print("Porcentaje de fraudes: %.2f%%" % (100 * df["isFraud"].mean()))
    print("Duplicados con TransactionID:", int(df.duplicated(subset="TransactionID").sum()))
    print("Nulos en las variables elegidas:", int(df.isna().sum().sum()))
    print("\nNulos principales (%):")
    print((df[VARIABLES].isna().mean() * 100).sort_values(ascending=False).head(10).round(2))

    n_muestra, _, _, _ = parametros_ejecucion(prueba)
    if len(df) < n_muestra:
        raise ValueError(f"Dataset con {len(df)} filas, menor a la muestra de {n_muestra}.")
    # Misma decision del grupo: submuestra aleatoria estratificada ANTES del split.
    df_muestra, _ = train_test_split(df[VARIABLES + ["isFraud"]],
                                    train_size=n_muestra,
                                    stratify=df["isFraud"], random_state=42)
    del df
    df_muestra = df_muestra.reset_index(drop=True)
    if not prueba and len(df_muestra) <= 100_000:
        raise ValueError("La pauta exige mas de 100.000 registros despues de limpiar.")

    print("\n2. EXPLORACION Y PIPELINE")
    print("Muestra estratificada:", df_muestra.shape)
    print("Clase 0/1 (%):")
    print((df_muestra["isFraud"].value_counts(normalize=True) * 100).round(3))
    print("Variable objetivo: isFraud (1 = fraude)")
    print("Predictoras seleccionadas:", len(VARIABLES))
    print("Monto de transaccion:")
    print(df_muestra["TransactionAmt"].describe().round(2))
    df_muestra.to_csv(salida / "Muestra_estratificada.csv", index=False)
    print("Muestra guardada en:", salida / "Muestra_estratificada.csv")
    return df_muestra

def particion(prueba=False):
    """Particion estratificada; TEST permanece reservado hasta evaluar."""
    _, salida, _ = carpetas(prueba)
    archivo = salida / "Muestra_estratificada.csv"
    if not archivo.exists():
        raise FileNotFoundError("Ejecuta primero: python Certamen1.py --etapa datos")
    df = pd.read_csv(archivo)
    X, y = df[VARIABLES], df["isFraud"].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=SEMILLA
    )
    return X_train, X_test, y_train, y_test

In [ ]:
if MODO == 'entrenar':
    df_muestra = cargar_datos()
    display(df_muestra.head())
    print('Distribución de fraude (%):')
    print((df_muestra['isFraud'].value_counts(normalize=True) * 100).round(2))
else:
    print('Se omite: resultados recuperados del ZIP.')

## 2. Preprocesamiento y partición de datos

In [ ]:
def crear_preprocesador(X_train):
    """El profesor: imputacion dentro de ColumnTransformer y Pipeline."""
    num = X_train.select_dtypes(include=np.number).columns.tolist()
    cat = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    print(f"Predictores: {len(num)} numericos y {len(cat)} categoricos")
    prep_num = PipelineSK([("imputar", SimpleImputer(strategy="median"))])
    prep_cat = PipelineSK([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    return ColumnTransformer([
        ("numericas", prep_num, num), ("categoricas", prep_cat, cat)
    ])

In [ ]:
if MODO == 'entrenar':
    X_train, X_test, y_train, y_test = particion()
    preprocesador = crear_preprocesador(X_train)
    print('Train:', X_train.shape, '| Test:', X_test.shape)
    print('Validación: 3 folds estratificados')
else:
    print('Se omite: resultados recuperados del ZIP.')

## 3. Etapa A — Modelos con SMOTE

In [ ]:
def crear_modelo(nombre, estrategia, proporcion, preprocesador):
    """3. SMOTE / 4. Pesos. Misma familia y grillas del notebook grupal."""
    pesos = estrategia == "Pesos"
    if nombre == "Decision Tree":
        estimador = DecisionTreeClassifier(random_state=SEMILLA,
                                           class_weight="balanced" if pesos else None)
        grilla = {"modelo__max_depth": [5, 10, 15, 20, None],
                  "modelo__min_samples_leaf": [1, 5, 10, 20]}
    elif nombre == "Random Forest":
        estimador = RandomForestClassifier(random_state=SEMILLA, n_jobs=1,
                                          class_weight="balanced" if pesos else None)
        grilla = {"modelo__n_estimators": [100, 150, 200],
                  "modelo__max_depth": [10, 20, None],
                  "modelo__min_samples_leaf": [1, 5, 10]}
    elif nombre == "XGBoost":
        estimador = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                  tree_method="hist", random_state=SEMILLA, n_jobs=1,
                                  scale_pos_weight=proporcion if pesos else 1)
        grilla = {"modelo__n_estimators": [100, 200, 300],
                  "modelo__max_depth": [3, 5, 7],
                  "modelo__learning_rate": [0.05, 0.10, 0.20]}
    elif nombre == "AdaBoost":
        # Mantener max_depth=1 en AMBAS estrategias. Diagnostico real del grupo:
        # class_weight='balanced' en el stump fallo en 20/30 ajustes CV.
        # Para la estrategia Pesos, aplicar sample_weight en AdaBoost.fit,
        # calculado SOLO con y de entrenamiento (nunca con el conjunto TEST).
        # NOTA: esta solucion ponderada NO cumple literalmente el requisito
        # de class_weight='balanced' dentro del arbol: documentar la excepcion.
        arbol = DecisionTreeClassifier(max_depth=1, random_state=SEMILLA)
        estimador = AdaBoostClassifier(estimator=arbol, random_state=SEMILLA)
        grilla = {"modelo__n_estimators": [50, 100, 150, 200],
                  "modelo__learning_rate": [0.05, 0.10, 0.50, 1.0]}
    elif nombre == "LightGBM":
        estimador = LGBMClassifier(objective="binary", random_state=SEMILLA,
                                   n_jobs=1, verbosity=-1,
                                   class_weight="balanced" if pesos else None)
        grilla = {"modelo__n_estimators": [100, 200, 300],
                  "modelo__max_depth": [5, 10, -1],
                  "modelo__learning_rate": [0.05, 0.10, 0.20]}
    else:
        raise ValueError(f"Modelo no reconocido: {nombre}")

    pasos = [("prep", clone(preprocesador))]
    if estrategia == "SMOTE":
        pasos.append(("smote", SMOTE(random_state=SEMILLA)))
    pasos.append(("modelo", estimador))
    return Pipeline(pasos), grilla

def id_modelo(nombre, estrategia):
    return f"{nombre.replace(' ', '')}_{estrategia}"

def entrenar_modelos(prueba=False, estrategia_unica=None):
    """3-4. Optimiza y entrena; seleccion solo con F1 CV. No consulta TEST."""
    _, salida, modelos_dir = carpetas(prueba)
    X_train, _, y_train, _ = particion(prueba)
    prep = crear_preprocesador(X_train)
    _, n_busqueda, n_iteraciones, n_folds = parametros_ejecucion(prueba)
    X_bus, _, y_bus, _ = train_test_split(X_train, y_train,
                                         train_size=min(n_busqueda, len(X_train)-1),
                                         stratify=y_train, random_state=SEMILLA)
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEMILLA)
    # Ponderacion de XGBoost calculada solo con TRAIN (no con el conjunto TEST).
    ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    tabla = []

    estrategias = [estrategia_unica] if estrategia_unica else ESTRATEGIAS
    for estrategia in estrategias:
        for nombre in NOMBRES:
            codigo = id_modelo(nombre, estrategia)
            ruta_modelo = modelos_dir / f"{codigo}.joblib"
            ruta_info = modelos_dir / f"{codigo}.json"
            if ruta_modelo.exists() and ruta_info.exists():
                print(f"[RECUPERADO] {codigo}: entrenamiento ya guardado")
                registro = json.loads(ruta_info.read_text(encoding="utf-8"))
                tabla.append(registro)
                continue

            print(f"\n{'='*60}\nEntrenando: {nombre} | {estrategia}")
            pipeline, grilla = crear_modelo(nombre, estrategia, ratio, prep)
            excepcion_pauta = "Ninguna"
            tiempo_acumulado = 0.0
            mejor = None
            es_ada_pesos = nombre == "AdaBoost" and estrategia == "Pesos"
            parametros_fit_cv = {}
            parametros_fit_train = {}
            if es_ada_pesos:
                # Replica la alternativa viable del diagnostico V2 (10/10 en CV).
                # Los pesos se calculan unicamente con las etiquetas de TRAIN.
                # RandomizedSearchCV distribuye el vector sample_weight entre folds.
                parametros_fit_cv = {
                    "modelo__sample_weight": compute_sample_weight(
                        class_weight="balanced", y=y_bus
                    )
                }
                parametros_fit_train = {
                    "modelo__sample_weight": compute_sample_weight(
                        class_weight="balanced", y=y_train
                    )
                }
                excepcion_pauta = (
                    "AdaBoost Pesos usa sample_weight balanceado en AdaBoost.fit; "
                    "arbol base max_depth=1, class_weight=None (no literal de pauta)"
                )
                print("NOTA ACADEMICA: AdaBoost con arbol max_depth=1 y pesos de "
                      "muestra; el arbol no tiene class_weight='balanced'.")

            errores = []
            for nro_intento, grilla_actual in enumerate([grilla]):
                busqueda = RandomizedSearchCV(
                    estimator=pipeline, param_distributions=grilla_actual,
                    n_iter=n_iteraciones, scoring="f1", cv=cv,
                    random_state=SEMILLA, n_jobs=N_JOBS, verbose=1,
                    error_score=np.nan, refit=False
                )
                inicio = time.perf_counter()
                busqueda.fit(X_bus, y_bus, **parametros_fit_cv)
                tiempo_acumulado += time.perf_counter() - inicio
                cv_resultados = pd.DataFrame(busqueda.cv_results_)
                validos = cv_resultados[cv_resultados["mean_test_score"].notna()].copy()
                fallidos = int(len(cv_resultados) - len(validos))
                if fallidos:
                    print(f"ADVERTENCIA: {fallidos} configuraciones CV fallidas en {codigo}.")
                if validos.empty:
                    errores.append(f"Intento {nro_intento+1}: sin parametros validos en CV")
                    continue
                # Intentar reentrenar segun el orden de F1 CV, por si un candidato
                # valido en los folds falla al ajustar TODO train.
                validos = validos.sort_values("mean_test_score", ascending=False)
                for _, candidato in validos.iterrows():
                    params_candidato = candidato["params"]
                    modelo_candidato = clone(pipeline).set_params(**params_candidato)
                    inicio_fit = time.perf_counter()
                    try:
                        modelo_candidato.fit(X_train, y_train, **parametros_fit_train)
                    except ValueError as exc:
                        errores.append(f"Fallo reentrenamiento: {str(exc)[:150]}")
                        continue
                    mejor = modelo_candidato
                    segundos_entrenamiento = time.perf_counter() - inicio_fit
                    mejor_param = params_candidato
                    mejor_cv = float(candidato["mean_test_score"])
                    mejor_std = float(candidato["std_test_score"])
                    break
                if mejor is not None:
                    break

            if mejor is None:
                raise RuntimeError(
                    f"No fue posible entrenar {codigo}. Errores: {errores}. "
                    "No se completara una tabla de 10 modelos con resultados inventados."
                )
            segundos_busqueda = tiempo_acumulado
            print("F1 CV:", round(mejor_cv, 4), "| parametros:", mejor_param)
            registro = {
                "Modelo": nombre, "Estrategia": estrategia,
                "F1 CV": mejor_cv, "Desv. F1 CV": mejor_std,
                "Tiempo busqueda (s)": segundos_busqueda,
                "Tiempo entrenamiento (s)": segundos_entrenamiento,
                "Configuraciones fallidas": fallidos,
                "Excepcion pauta": excepcion_pauta,
                "Mejores parametros": str(mejor_param),
                "Parametros modelo (json)": {k.removeprefix("modelo__"): v
                                           for k, v in mejor_param.items()}
            }
            # Guardar checkpoints: si cae VS Code, se puede continuar.
            joblib.dump(mejor, ruta_modelo, compress=3)
            ruta_info.write_text(json.dumps(registro, ensure_ascii=False, indent=2),
                                encoding="utf-8")
            tabla.append(registro)
            pd.DataFrame(tabla).to_csv(salida / "Tabla_CV_parcial.csv", index=False)

    # Reconstruimos la tabla con modelos que ya quedaron guardados en cada etapa.
    todos = []
    for estrategia in ESTRATEGIAS:
        for nombre in NOMBRES:
            ruta_info = modelos_dir / f"{id_modelo(nombre, estrategia)}.json"
            if ruta_info.exists():
                todos.append(json.loads(ruta_info.read_text(encoding="utf-8")))

    df = pd.DataFrame(todos).sort_values("F1 CV", ascending=False).reset_index(drop=True)
    if len(df) == 10:
        df.to_csv(salida / "Tabla_CV_10_combinaciones.csv", index=False)
        print("\nComparación final por F1 CV:")
        print(df[["Modelo", "Estrategia", "F1 CV"]].to_string(index=False))
        print("Ganador por CV:", df.iloc[0]["Modelo"], df.iloc[0]["Estrategia"])
    else:
        df.to_csv(salida / "Tabla_CV_parcial.csv", index=False)
        print(f"Etapa {estrategia_unica or 'modelos'} completada ({len(df)}/10 modelos).")
    return df

In [ ]:
if MODO == 'entrenar':
    # Etapa A: Decision Tree, Random Forest, XGBoost, AdaBoost y LightGBM
    resultados_A = entrenar_modelos(estrategia_unica='SMOTE')
    display(resultados_A[resultados_A['Estrategia'] == 'SMOTE'][['Modelo', 'F1 CV']])
else:
    print('Se omite: resultados recuperados del ZIP.')

## 4. Etapa B — Modelos con pesos

In [ ]:
if MODO == 'entrenar':
    # Etapa B: los mismos cinco algoritmos, sin SMOTE
    resultados_B = entrenar_modelos(estrategia_unica='Pesos')
    tabla_cv = pd.read_csv(RESULTADOS / 'Tabla_CV_10_combinaciones.csv')
    display(tabla_cv[['Modelo', 'Estrategia', 'F1 CV', 'Desv. F1 CV']])
else:
    tabla_cv = pd.read_csv(RESULTADOS / 'Tabla_CV_10_combinaciones.csv')
    display(tabla_cv[['Modelo', 'Estrategia', 'F1 CV', 'Desv. F1 CV']])

## 5. Comparación y evaluación de modelos

In [ ]:
def evaluar_modelos(prueba=False):
    """5. Evaluacion en test una vez fijados los 10 modelos y ganador por CV."""
    _, salida, modelos_dir = carpetas(prueba)
    tabla_cv = salida / "Tabla_CV_10_combinaciones.csv"
    if not tabla_cv.exists():
        raise FileNotFoundError("Primero ejecuta --etapa modelos (10 modelos completos).")
    tabla = pd.read_csv(tabla_cv)
    if len(tabla) != 10:
        raise RuntimeError("Faltan combinaciones: deben ser exactamente 10.")
    X_train, X_test, y_train, y_test = particion(prueba)
    resultados = []
    ganador = tabla.iloc[0]  # FIJADO ANTES DE ABRIR EL TEST
    for _, fila in tabla.iterrows():
        nombre, estrategia = fila["Modelo"], fila["Estrategia"]
        archivo = modelos_dir / f"{id_modelo(nombre, estrategia)}.joblib"
        modelo = joblib.load(archivo)
        pred_train = modelo.predict(X_train)
        pred_test = modelo.predict(X_test)
        prob_test = modelo.predict_proba(X_test)[:, 1]
        resultados.append({
            "Modelo": nombre, "Estrategia": estrategia,
            "F1 train": f1_score(y_train, pred_train),
            "F1 test": f1_score(y_test, pred_test),
            "Brecha train-test": f1_score(y_train, pred_train) - f1_score(y_test, pred_test),
            "Precision": precision_score(y_test, pred_test, zero_division=0),
            "Recall": recall_score(y_test, pred_test, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, prob_test)
        })
        del modelo
    tabla_final = tabla.merge(pd.DataFrame(resultados), on=["Modelo", "Estrategia"])
    tabla_final = tabla_final[[
        "Modelo", "Estrategia", "Mejores parametros", "F1 train", "F1 CV",
        "Desv. F1 CV", "F1 test", "Brecha train-test", "Recall", "Precision",
        "ROC-AUC", "Tiempo busqueda (s)", "Tiempo entrenamiento (s)",
        "Configuraciones fallidas", "Excepcion pauta"
    ]].sort_values("F1 CV", ascending=False)
    tabla_final.to_csv(salida / "Comparacion_10_Modelos.csv", index=False)
    print("\n5. TABLA COMPARATIVA FINAL")
    print(tabla_final.round(4).to_string(index=False))
    print("\nGanador por F1 CV:", ganador["Modelo"], "|", ganador["Estrategia"])
    return tabla_final

In [ ]:
if MODO == 'entrenar':
    tabla_final = evaluar_modelos()
else:
    tabla_final = pd.read_csv(RESULTADOS / 'Comparacion_10_Modelos.csv')
display(tabla_final[['Modelo', 'Estrategia', 'F1 CV', 'F1 test', 'Recall', 'Precision', 'ROC-AUC']])

## 6. Costos de clasificación y umbrales

In [ ]:
def costo_total(y_real, y_pred, costo_fp, costo_fn):
    """6. Matriz de costos (TP y TN tienen costo cero en este supuesto)."""
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return int(fp*costo_fp + fn*costo_fn)

def scorer_costo(costo_fp, costo_fn):
    """Puntaje mayor = costo monetario menor."""
    return make_scorer(lambda yt, yp: -costo_total(yt, yp, costo_fp, costo_fn))

def fila_costos(nombre, y_real, y_pred, costo_fp, costo_fn, costo_ref):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    costo = costo_total(y_real, y_pred, costo_fp, costo_fn)
    return {"Politica": nombre, "TN": int(tn), "FP": int(fp),
            "FN": int(fn), "TP": int(tp),
            "Precision": precision_score(y_real, y_pred, zero_division=0),
            "Recall": recall_score(y_real, y_pred, zero_division=0),
            "F1": f1_score(y_real, y_pred, zero_division=0),
            "Costo total (CLP)": costo, "Ahorro vs 0.5 (CLP)": int(costo_ref-costo)}

def umbral_por_probabilidades(y_real, probabilidades, costo_fp, costo_fn, pasos=501):
    """Umbral empírico de exploracion sobre probabilidades OOF SOLO DE TRAIN."""
    umbrales = np.linspace(0, 1, pasos)
    costes = [costo_total(y_real, probabilidades >= t, costo_fp, costo_fn)
              for t in umbrales]
    indice = int(np.argmin(costes))
    return float(umbrales[indice])

def evaluar_costos(prueba=False):
    """6-8. Costos, threshold teorico/empirico, sensibilidad sin usar test al optimizar."""
    _, salida, modelos_dir = carpetas(prueba)
    tabla = salida / "Comparacion_10_Modelos.csv"
    if not tabla.exists():
        raise FileNotFoundError("Primero ejecuta --etapa evaluar.")
    tabla = pd.read_csv(tabla)
    X_train, X_test, y_train, y_test = particion(prueba)
    preprocesador = crear_preprocesador(X_train)
    ganador = tabla.iloc[0]
    ganador_nombre = ganador["Modelo"]
    # En esta parte, segun pauta, se conservan hiperparametros pero se retira SMOTE/pesos.
    modelo_ganador = joblib.load(modelos_dir / f"{id_modelo(ganador_nombre, ganador['Estrategia'])}.joblib")
    # Rescatar SOLO los hiperparametros elegidos por CV, no parametros accesorios
    # del estimador ajustado, que pueden cambiar entre versiones de bibliotecas.
    hiperparametros = modelo_ganador.named_steps["modelo"].get_params()
    hiper_seleccionados = {k: hiperparametros[k] for k in
                          ["max_depth", "min_samples_leaf", "n_estimators", "learning_rate"]
                          if k in hiperparametros}
    pipe_real, _ = crear_modelo(ganador_nombre, "Real", proporcion=1,
                               preprocesador=preprocesador)
    # Mantener la parametrizacion del modelo escogido por CV (sin sus pesos).
    ajustes = {f"modelo__{k}": v for k, v in hiper_seleccionados.items()
               if k in pipe_real.named_steps["modelo"].get_params()}
    for k in ["modelo__class_weight", "modelo__scale_pos_weight", "modelo__is_unbalance"]:
        ajustes.pop(k, None)
    # AdaBoost: el arbol interno del ganador puede tener class_weight.
    if ganador_nombre == "AdaBoost":
        ajustes.pop("modelo__estimator", None)
    pipe_real.set_params(**ajustes)
    print("\n7. Reentrenando modelo con proporcion real, sin SMOTE ni pesos")
    pipe_real.fit(X_train, y_train)
    joblib.dump(pipe_real, modelos_dir / "Modelo_probabilidades_reales.joblib", compress=3)
    prob_test = pipe_real.predict_proba(X_test)[:, 1]
    _, n_busqueda, _, n_folds = parametros_ejecucion(prueba)
    X_bus, _, y_bus, _ = train_test_split(X_train, y_train,
                                         train_size=min(n_busqueda, len(X_train)-1),
                                         stratify=y_train, random_state=SEMILLA)
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEMILLA)

    # OOF evita usar probabilidades de entrenamiento vistas por el modelo.
    print("Calculando probabilidades out-of-fold para sensibilidad y dashboard...")
    prob_cv = cross_val_predict(clone(pipe_real), X_bus, y_bus, cv=cv,
                                method="predict_proba", n_jobs=N_JOBS)[:, 1]
    pd.DataFrame({"y_real": y_bus.to_numpy(), "proba": prob_cv}).to_csv(
        salida / "Probabilidades_CV_train.csv", index=False)
    brier = brier_score_loss(y_bus, prob_cv)
    print("Brier en submuestra de entrenamiento por CV:", round(brier, 5))
    print("AVISO: quitar SMOTE/pesos no garantiza calibracion perfecta.")

    # TunedThresholdClassifierCV, como en notebook de clasificacion sensible al costo.
    umbral_teorico = C_FP / (C_FP + C_FN)
    escenarios = [("Bajo", C_FP, 10_000), ("Moderado", C_FP, 25_000),
                  ("Base", C_FP, C_FN), ("Alto", C_FP, 100_000),
                  ("Extremo", C_FP, 200_000)]
    tabla_escenarios = []
    umbral_empirico = None
    for etiqueta, cfp, cfn in escenarios:
        umbral_teo = cfp/(cfp+cfn)
        print(f"Optimizando costo CV: {etiqueta} (FP={cfp:,}, FN={cfn:,})")
        tuned = TunedThresholdClassifierCV(
            estimator=clone(pipe_real), scoring=scorer_costo(cfp, cfn),
            response_method="predict_proba", thresholds=100,
            cv=cv, n_jobs=N_JOBS, refit=True
        )
        tuned.fit(X_bus, y_bus)  # SOLO TRAIN (submuestra estratificada)
        emp = float(tuned.best_threshold_)
        if etiqueta == "Base":
            umbral_empirico = emp
        pred = (prob_test >= emp).astype(int)
        referencia = costo_total(y_test, prob_test >= 0.5, cfp, cfn)
        tabla_escenarios.append({
            "Escenario": etiqueta, "C_FP": cfp, "C_FN": cfn,
            "Relacion FN/FP": cfn/cfp, "Umbral teorico": umbral_teo,
            "Umbral empirico CV": emp, "FP": int(((pred==1)&(y_test==0)).sum()),
            "FN": int(((pred==0)&(y_test==1)).sum()),
            "Costo total (CLP)": costo_total(y_test, pred, cfp, cfn),
            "Ahorro vs 0.5 (CLP)": referencia-costo_total(y_test, pred, cfp, cfn)
        })
    sensibilidad = pd.DataFrame(tabla_escenarios)
    sensibilidad.to_csv(salida / "Sensibilidad_Costos.csv", index=False)

    # Comparar la etapa B: mejor modelo de estrategia ponderada por F1 CV.
    ganador_pesos = tabla[tabla["Estrategia"] == "Pesos"].sort_values("F1 CV", ascending=False).iloc[0]
    pipe_pesos = joblib.load(modelos_dir / f"{id_modelo(ganador_pesos['Modelo'], 'Pesos')}.joblib")
    prob_pesos = pipe_pesos.predict_proba(X_test)[:, 1]
    pred_pesos = pipe_pesos.predict(X_test)
    costo_base = costo_total(y_test, prob_test >= 0.5, C_FP, C_FN)
    politicas = {
        "Todo 0": np.zeros(len(y_test), dtype=int),
        "Umbral 0.5": (prob_test >= 0.5).astype(int),
        "Umbral teorico": (prob_test >= umbral_teorico).astype(int),
        "Umbral empirico CV": (prob_test >= umbral_empirico).astype(int),
        f"{ganador_pesos['Modelo']} Pesos": pred_pesos
    }
    tabla_politicas = pd.DataFrame([
        fila_costos(n, y_test, p, C_FP, C_FN, costo_base)
        for n, p in politicas.items()
    ])
    tabla_politicas.to_csv(salida / "Comparacion_Costos.csv", index=False)
    (salida / "Configuracion_Costos.json").write_text(
        json.dumps({"C_FP": C_FP, "C_FN": C_FN,
                    "Umbral teorico": float(umbral_teorico),
                    "Umbral empirico CV": float(umbral_empirico),
                    "Modelo ganador": str(ganador_nombre),
                    "Estrategia ganadora": str(ganador["Estrategia"]),
                    "Estrategia pesos comparada": str(ganador_pesos["Modelo"]),
                    "Muestra": 250000 if not prueba else 4000,
                    "Prueba rapida": bool(prueba)}, ensure_ascii=False, indent=2),
        encoding="utf-8")
    pd.DataFrame({"y_real": y_test.to_numpy(), "proba": prob_test,
                  "proba_pesos": prob_pesos}).to_csv(
                      salida / "Probabilidades_Test.csv", index=False)
    print("\n7. COMPARACION ECONOMICA EN TEST")
    print(tabla_politicas.round(4).to_string(index=False))
    print("\n8. SENSIBILIDAD ECONOMICA EN TEST (umbrales elegidos en CV)")
    print(sensibilidad.round(4).to_string(index=False))
    print("\nResultados guardados:", salida)

In [ ]:
# Costos asumidos: FP = $5.000 y FN = $50.000 (CLP)
if MODO == 'entrenar':
    evaluar_costos()
comparacion_costos = pd.read_csv(RESULTADOS / 'Comparacion_Costos.csv')
config_costos = json.loads((RESULTADOS / 'Configuracion_Costos.json').read_text(encoding='utf-8'))
display(comparacion_costos)
print('Umbral teórico:', round(config_costos['Umbral teorico'], 4))
print('Umbral empírico:', round(config_costos['Umbral empirico CV'], 4))

## 7. Análisis de sensibilidad

In [ ]:
sensibilidad = pd.read_csv(RESULTADOS / 'Sensibilidad_Costos.csv')
display(sensibilidad)

## 8. Gráficos de resultados

In [ ]:
# Formato común para los gráficos
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

GRAFICOS = RESULTADOS / 'graficos'
GRAFICOS.mkdir(exist_ok=True)
COLOR_1 = '#213B64'
COLOR_2 = '#148D91'
COLOR_3 = '#D89C3A'

def guardar(nombre):
    plt.savefig(GRAFICOS / f'{nombre}.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()

def clp(valor, pos=None):
    return f'${valor/1_000_000:,.0f}M'

In [ ]:
# 1. Distribución de clases
if MODO == 'entrenar':
    cuentas = df_muestra['isFraud'].value_counts().sort_index()
    subtitulo = 'muestra estratificada (250.000)'
else:
    # En modo resultados no se necesita subir la muestra completa.
    cuentas = pd.read_csv(RESULTADOS / 'Probabilidades_Test.csv')['y_real'].value_counts().sort_index()
    subtitulo = 'TEST reservado (50.000)'
fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(['Legítimas', 'Fraudes'], cuentas.values, color=[COLOR_1, COLOR_2])
ax.bar_label(barras, labels=[f'{v:,}'.replace(',', '.') for v in cuentas.values], padding=4)
ax.set(title='Distribución de clases — ' + subtitulo, ylabel='Transacciones')
ax.spines[['top','right']].set_visible(False)
guardar('01_clases')

In [ ]:
# 2. F1 por combinación (el ganador se define por validación cruzada)
r = tabla_cv.sort_values('F1 CV')
etiquetas = r['Modelo'] + ' - ' + r['Estrategia']
colores = [COLOR_2 if i == r.index[-1] else COLOR_1 for i in r.index]
fig, ax = plt.subplots(figsize=(9, 5.5))
barras = ax.barh(etiquetas, r['F1 CV'], color=colores)
ax.bar_label(barras, fmt='%.3f', padding=4)
ax.set_xlim(0, max(0.55, r['F1 CV'].max() * 1.15))
ax.set_xlabel('F1 CV')
ax.set_title('Comparación de las 10 combinaciones')
ax.spines[['top', 'right']].set_visible(False)
guardar('02_modelos_F1_CV')

In [ ]:
# 3. F1 train vs test
r = tabla_final.sort_values('F1 CV')
etiquetas = r['Modelo'] + ' - ' + r['Estrategia']
y = np.arange(len(r))
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(y - 0.18, r['F1 train'], height=0.35, color=COLOR_1, label='Train')
ax.barh(y + 0.18, r['F1 test'], height=0.35, color=COLOR_2, label='Test')
ax.set_yticks(y, etiquetas)
ax.set_title('F1 de entrenamiento vs prueba')
ax.set_xlabel('F1')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
guardar('03_F1_train_test')

In [ ]:
# 4. Costos de las políticas evaluadas
r = comparacion_costos.sort_values('Costo total (CLP)', ascending=True)
fig, ax = plt.subplots(figsize=(10, 4.8))
barras = ax.barh(r['Politica'], r['Costo total (CLP)'], color=COLOR_1)
ax.bar_label(barras, labels=[f'${v/1e6:.2f}M' for v in r['Costo total (CLP)']], padding=4)
ax.set_xlim(0, r['Costo total (CLP)'].max() * 1.2)
ax.set_title('Costo total por política de clasificación (TEST)')
ax.set_xlabel('Pesos chilenos (millones)')
ax.xaxis.set_major_formatter(FuncFormatter(clp))
ax.spines[['top', 'right']].set_visible(False)
guardar('04_costos_por_politica')

In [ ]:
# 5. Matrices de confusión del modelo probabilístico
from sklearn.metrics import ConfusionMatrixDisplay
probs = pd.read_csv(RESULTADOS / 'Probabilidades_Test.csv')
y_test_vec = probs['y_real'].to_numpy()
prob_test = probs['proba'].to_numpy()
umbral_teo = float(config_costos['Umbral teorico'])
umbral_emp = float(config_costos['Umbral empirico CV'])
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (nombre, umbral) in zip(axs, [('0,5', 0.5), ('Teórico', umbral_teo), ('Empírico CV', umbral_emp)]):
    pred = (prob_test >= umbral).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test_vec, pred, labels=[0, 1],
                                            display_labels=['Legítima', 'Fraude'],
                                            cmap='Blues', colorbar=False, ax=ax, values_format='d')
    ax.set_title(f'Umbral {nombre}: {umbral:.3f}')
plt.tight_layout()
guardar('05_matrices_confusion')

In [ ]:
# 6. Costo vs umbral calculado en TRAIN (predicciones fuera de fold)
prob_cv = pd.read_csv(RESULTADOS / 'Probabilidades_CV_train.csv')
y_cv = prob_cv['y_real'].to_numpy()
p_cv = prob_cv['proba'].to_numpy()
umbrales = np.linspace(0.01, 0.50, 100)
costos_cv = [costo_total(y_cv, p_cv >= u, C_FP, C_FN) for u in umbrales]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(umbrales, np.array(costos_cv) / 1e6, color=COLOR_1, linewidth=2)
ax.axvline(umbral_teo, color=COLOR_3, linestyle='--', label=f'Teórico: {umbral_teo:.3f}')
ax.axvline(umbral_emp, color=COLOR_2, linestyle='--', label=f'Empírico CV: {umbral_emp:.3f}')
ax.set(title='Costo vs umbral (validación cruzada en TRAIN)', xlabel='Umbral', ylabel='Costo (millones de CLP)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
guardar('06_costo_vs_umbral_train')

In [ ]:
# 7. Sensibilidad de costos por escenario
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sensibilidad['Relacion FN/FP'], sensibilidad['Costo total (CLP)'] / 1e6,
        marker='o', linewidth=2, color=COLOR_2)
for _, f in sensibilidad.iterrows():
    ax.annotate(f['Escenario'], (f['Relacion FN/FP'], f['Costo total (CLP)'] / 1e6),
                xytext=(4, 7), textcoords='offset points')
ax.set(title='Sensibilidad: costo según relación FN / FP',
       xlabel='Relación de costos FN / FP', ylabel='Costo (millones de CLP)')
ax.spines[['top', 'right']].set_visible(False)
guardar('07_sensibilidad')

## 9. Dashboard interactivo

In [ ]:
# Dashboard en Colab: probabilidades guardadas; no se reentrenan modelos.
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.ticker import FuncFormatter
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# Estos CSV fueron generados en las etapas anteriores usando TEST reservado y TRAIN CV.
datos_test = pd.read_csv(RESULTADOS / 'Probabilidades_Test.csv')
datos_cv = pd.read_csv(RESULTADOS / 'Probabilidades_CV_train.csv')
y_t = datos_test['y_real'].to_numpy(dtype=int)
p_t = datos_test['proba'].to_numpy(dtype=float)
y_cv = datos_cv['y_real'].to_numpy(dtype=int)
p_cv = datos_cv['proba'].to_numpy(dtype=float)
p_pesos = datos_test['proba_pesos'].to_numpy(dtype=float) if 'proba_pesos' in datos_test else None
umbral_emp_base = float(config_costos['Umbral empirico CV'])  # Fijado en TRAIN, costos base

# Controles principales
control_fp = widgets.IntSlider(value=5000, min=1000, max=20000, step=1000,
                               description='Costo FP', continuous_update=False,
                               style={'description_width': '90px'}, layout=widgets.Layout(width='380px'))
control_fn = widgets.IntSlider(value=50000, min=10000, max=200000, step=5000,
                               description='Costo FN', continuous_update=False,
                               style={'description_width': '90px'}, layout=widgets.Layout(width='380px'))
control_umbral = widgets.FloatSlider(value=0.5, min=0.01, max=0.99, step=0.01,
                                     readout_format='.2f', description='Manual',
                                     continuous_update=False, style={'description_width': '90px'},
                                     layout=widgets.Layout(width='380px'))
control_politica = widgets.Dropdown(options=['Umbral 0.5', 'Umbral teórico',
                                            'Umbral empírico CV (base)', 'Umbral manual'] +
                                           (['Modelo con pesos (0.5)'] if p_pesos is not None else []),
                                   value='Umbral empírico CV (base)', description='Política',
                                   style={'description_width': '90px'}, layout=widgets.Layout(width='380px'))

formato_clp = lambda n: '$' + f'{round(n):,}'.replace(',', '.')


def mostrar_dashboard(costo_fp, costo_fn, umbral, politica):
    teo = costo_fp / (costo_fp + costo_fn)
    # El umbral empírico permanece fijo: se optimizó antes sobre TRAIN para costos BASE.
    umbrales = {'Umbral 0.5': .5, 'Umbral teórico': teo,
                'Umbral empírico CV (base)': umbral_emp_base, 'Umbral manual': umbral}
    decisiones = {nombre: (p_t >= valor).astype(int) for nombre, valor in umbrales.items()}
    if p_pesos is not None:
        decisiones['Modelo con pesos (0.5)'] = (p_pesos >= .5).astype(int)

    resumen = []
    for nombre, prediccion in decisiones.items():
        tn, fp, fn, tp = confusion_matrix(y_t, prediccion, labels=[0, 1]).ravel()
        resumen.append({'Política': nombre, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                        'Precisión': precision_score(y_t, prediccion, zero_division=0),
                        'Recall': recall_score(y_t, prediccion, zero_division=0),
                        'Costo': fp*costo_fp + fn*costo_fn})
    tabla = pd.DataFrame(resumen)
    referencia = int(tabla.loc[tabla['Política']=='Umbral 0.5', 'Costo'].iloc[0])
    elegido = tabla.set_index('Política').loc[politica]
    ahorro = referencia - int(elegido['Costo'])

    print('Umbral teórico:', f'{teo:.4f}', '| Umbral empírico CV (costos base):', f'{umbral_emp_base:.4f}')
    print('Costo con umbral 0.5:', formato_clp(referencia),
          '| Costo política elegida:', formato_clp(elegido['Costo']),
          '| Ahorro:', formato_clp(ahorro))
    tabla_visible = tabla[['Política', 'FP', 'FN', 'TP', 'TN', 'Precisión', 'Recall', 'Costo']].copy()
    tabla_visible['Costo'] = tabla_visible['Costo'].map(formato_clp)
    display(tabla_visible.style.format({'Precisión':'{:.3f}', 'Recall':'{:.3f}'}))

    fig, ejes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={'width_ratios':[1,1.5]})
    # Matriz de confusión para la política seleccionada
    tn, fp, fn, tp = (int(elegido[col]) for col in ['TN', 'FP', 'FN', 'TP'])
    matriz = np.array([[tn, fp], [fn, tp]])
    ejes[0].imshow(matriz, cmap='Blues')
    for fila in range(2):
        for columna in range(2):
            valor = matriz[fila, columna]
            ejes[0].text(columna, fila, f'{valor:,}'.replace(',', '.'), ha='center', va='center',
                         color=('white' if valor > matriz.max()/2 else '#1f2937'), fontweight='bold', fontsize=12)
    ejes[0].set_xticks([0,1], ['Pred. 0', 'Pred. 1'])
    ejes[0].set_yticks([0,1], ['Real 0', 'Real 1'])
    ejes[0].set_title('Matriz de confusión — ' + politica, fontsize=11)
    ejes[0].set_xlabel('Predicción')
    ejes[0].set_ylabel('Real')

    # Comparación económica dinámica entre las políticas
    barras = ejes[1].barh(tabla['Política'], tabla['Costo']/1e6,
                         color=[COLOR_2 if nombre==politica else COLOR_1 for nombre in tabla['Política']])
    ejes[1].bar_label(barras, fmt='%.1f M', padding=3, fontsize=9)
    ejes[1].invert_yaxis()
    ejes[1].set_xlim(0, max(tabla['Costo']/1e6)*1.2)
    ejes[1].set_xlabel('Costo (millones de CLP)')
    ejes[1].set_title('Costo total por política — TEST', fontsize=11)
    ejes[1].spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()

    # La curva se calcula ÚNICAMENTE con predicciones OOF de TRAIN.
    puntos = np.linspace(0.01, 0.80, 130)
    costo_cv = np.array([((p_cv >= u) & (y_cv==0)).sum()*costo_fp +
                         ((p_cv < u) & (y_cv==1)).sum()*costo_fn for u in puntos])
    fig, ax = plt.subplots(figsize=(11, 3.3))
    ax.plot(puntos, costo_cv/1e6, color=COLOR_1, linewidth=2)
    ax.axvline(teo, color=COLOR_3, linestyle='--', label=f'Teórico ({teo:.3f})')
    ax.axvline(umbral_emp_base, color=COLOR_2, linestyle=':', label=f'Empírico CV base ({umbral_emp_base:.3f})')
    ax.axvline(umbral, color='#7E5AA6', linestyle='--', alpha=0.6, label=f'Manual ({umbral:.2f})')
    ax.set(xlabel='Umbral de clasificación', ylabel='Costo (millones de CLP)',
           title='Curva de costos — validación cruzada en TRAIN')
    ax.legend(loc='best', fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.show()
    print('Simulación para TEST; las políticas no se eligen mirando TEST. '
          'El umbral empírico CV corresponde a los costos base ($5.000 / $50.000).')

salida = widgets.interactive_output(
    mostrar_dashboard,
    {'costo_fp':control_fp, 'costo_fn':control_fn,
     'umbral':control_umbral, 'politica':control_politica}
)
display(widgets.VBox([
    widgets.HTML('<h3>Simulador económico de fraude</h3>'),
    widgets.HBox([widgets.VBox([control_fp,control_fn]),
                  widgets.VBox([control_umbral,control_politica])]), salida
]))

## 10. Guardar resultados y gráficos

In [ ]:
# Guardar gráficos y resultados para compartir sin repetir el entrenamiento
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

nombres = [
    'Tabla_CV_10_combinaciones.csv', 'Comparacion_10_Modelos.csv',
    'Comparacion_Costos.csv', 'Sensibilidad_Costos.csv',
    'Probabilidades_Test.csv', 'Probabilidades_CV_train.csv',
    'Configuracion_Costos.json'
]
ruta_zip = PROYECTO / 'Resultados_Certamen1.zip'
with ZipFile(ruta_zip, 'w', ZIP_DEFLATED) as archivo_zip:
    for nombre in nombres:
        ruta = RESULTADOS / nombre
        if not ruta.exists():
            raise FileNotFoundError('Falta ' + str(ruta))
        archivo_zip.write(ruta, arcname='resultados/' + nombre)
    for archivo in (RESULTADOS / 'graficos').glob('*.png'):
        archivo_zip.write(archivo, arcname='graficos/' + archivo.name)
print('Archivo listo:', ruta_zip)
files.download(str(ruta_zip))

**Nota:** Los costos FP y FN son supuestos del grupo. Para AdaBoost ponderado se utilizaron pesos de muestra (`sample_weight`) con árbol de profundidad 1; esto no equivale literalmente a `class_weight="balanced"` dentro del árbol. La selección por F1 CV y los umbrales ajustados no utilizan TEST.